# RAG LLM LawBot

## Install relevant packages

In [4]:
%%capture

!pip install unsloth
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -U sentence-transformers
!pip install -q langchain==0.1.20
!pip install -q langchain-community==0.0.38
!pip install -q chromadb==0.4.24
!pip install -q gradio==4.36.1
!pip install -q pymupdf==1.23.8
!pip install -q sentence-transformers==2.2.2
!pip install youtube_dl
!pip install whisper

## Import all relevant packages throughout this walkthrough

In [5]:
# Modules for fine-tuning
from unsloth import FastLanguageModel
import torch # Import PyTorch
from trl import SFTTrainer # Trainer for supervised fine-tuning (SFT)
from unsloth import is_bfloat16_supported # Checks if the hardware supports bfloat16 precision
# Hugging Face modules
from huggingface_hub import login # Lets you login to API
from transformers import TrainingArguments # Defines training hyperparameters
from datasets import load_dataset # Lets you load fine-tuning datasets
# Import weights and biases
import wandb
# Import kaggle secrets
from kaggle_secrets import UserSecretsClient

# Import necessary packages
import gradio as gr
import torch
import re
import os
from pathlib import Path

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

# Embedding generation using HuggingFace embeddings instead of Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings

# Import kaggle secrets for token management
from kaggle_secrets import UserSecretsClient

from transformers import pipeline

## Import all relevant packages
import torch
import re
import os
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import youtube_dl
import whisper

# Hugging Face modules
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Unsloth and model imports
from unsloth import FastLanguageModel
from transformers import pipeline

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document

# Alternative embedding using transformers directly
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-05-28 17:56:59.752685: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748455019.957841      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748455020.018268      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


## Create API keys and login to Hugging Face and Weights and Biases

In [6]:
user_secrets = UserSecretsClient()
hugging_face_token = user_secrets.get_secret("HF_TOKEN_DEEPSEEK")
wnb_token = user_secrets.get_secret("wnb_token")

login(hugging_face_token)

wandb.login(key=wnb_token)
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-Distill-Llama-8B on LawBot', 
    job_type="training", 
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: naufalkr394 (naufalkr394-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Loading Model and the Tokenizer

In [7]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hugging_face_token,
)

==((====))==  Unsloth 2025.5.8: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

## Testing Model on a law use-case before fine-tuning

In [8]:
prompt_style = """
Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut.
Tuliskan respons yang menyelesaikan permintaan dengan tepat.
Sebelum menjawab, pikirkan dengan cermat pertanyaan tersebut dan buatlah rangkaian pemikiran langkah demi langkah untuk memastikan respons yang logis dan akurat.

### Instruksi:
Anda adalah seorang ahli hukum dengan pengetahuan tingkat lanjut dalam penalaran hukum, analisis kasus, dan penyusunan dokumen hukum. 
Jawablah pertanyaan hukum berikut ini dengan tepat, berdasarkan peraturan perundang-undangan yang berlaku dan preseden hukum yang relevan.

### Pertanyaan:
{}

### Jawaban:
<think>
{}
"""

### Running inference on the model


In [9]:
question = """Apa arti dari “berada di bawah Presiden” dalam konteks TNI?"""

FastLanguageModel.for_inference(model)

inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)

print(response[0].split("### Response:")[0])

<｜begin▁of▁sentence｜>
Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut.
Tuliskan respons yang menyelesaikan permintaan dengan tepat.
Sebelum menjawab, pikirkan dengan cermat pertanyaan tersebut dan buatlah rangkaian pemikiran langkah demi langkah untuk memastikan respons yang logis dan akurat.

### Instruksi:
Anda adalah seorang ahli hukum dengan pengetahuan tingkat lanjut dalam penalaran hukum, analisis kasus, dan penyusunan dokumen hukum. 
Jawablah pertanyaan hukum berikut ini dengan tepat, berdasarkan peraturan perundang-undangan yang berlaku dan preseden hukum yang relevan.

### Pertanyaan:
Apa arti dari “berada di bawah Presiden” dalam konteks TNI?

### Jawaban:
<think>

Anda adalah seorang ahli hukum dengan pengetahuan tingkat lanjut dalam penalaran hukum, analisis kasus, dan penyusunan dokumen hukum. Anda harus memberikan jawaban yang tepat berdasarkan peraturan perundang-undangan yang berlaku dan preseden hukum 

## **RAG Implementation**

In [21]:
# Create text generation pipeline
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=2048,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
)

Device set to use cuda:0


In [11]:
## Custom Embedding Class (Alternative to sentence-transformers)
class SimpleEmbeddings:
    def __init__(self, model_name="distilbert-base-uncased"):
        print(f"Loading embedding model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()
        
    def embed_documents(self, texts):
        """Embed a list of documents"""
        embeddings = []
        for text in texts:
            embedding = self._get_embedding(text)
            embeddings.append(embedding)
        return embeddings
    
    def embed_query(self, text):
        """Embed a single query"""
        return self._get_embedding(text)
    
    def _get_embedding(self, text):
        """Get embedding for a single text"""
        try:
            # Truncate text if too long
            text = text[:512]
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Use mean pooling of last hidden states
                embeddings = outputs.last_hidden_state.mean(dim=1)
                # Normalize
                embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)
                
            return embeddings.squeeze().numpy()
        except Exception as e:
            print(f"Error getting embedding: {e}")
            # Return zero vector as fallback
            return np.zeros(768)

In [12]:
class MultiSourceRAG:
    def __init__(self):
        self.documents = []
        self.vectorstore = None
        self.retriever = None
        self.embeddings = SimpleEmbeddings()
        self.document_embeddings = []
        self.chunk_texts = []
        
    def process_pdf(self, pdf_path):
        """Process PDF documents (books, analysis about UU TNI)"""
        print(f"\nProcessing PDF: {pdf_path}")
        try:
            loader = PyMuPDFLoader(pdf_path)
            docs = loader.load()
            
            # Add metadata to distinguish source
            for doc in docs:
                doc.metadata['source_type'] = 'pdf'
                doc.metadata['source_file'] = os.path.basename(pdf_path)
            
            self.documents.extend(docs)
            print(f"PDF processed: {len(docs)} pages added")
            return True
        except Exception as e:
            print(f"Error processing PDF: {e}")
            return False
    
    def scrape_web_articles(self, urls):
        """Scrape web articles/news about UU TNI"""
        print(f"\nScraping {len(urls)} web articles...")
        for i, url in enumerate(urls):
            try:
                print(f"  Scraping article {i+1}: {url[:50]}...")
                response = requests.get(url, timeout=10)
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Extract title and content (adjust selectors as needed)
                title = soup.find('title')
                title_text = title.get_text() if title else "Unknown Title"
                
                # Remove script and style elements
                for script in soup(["script", "style"]):
                    script.decompose()
                
                # Get text content
                content = soup.get_text()
                # Clean up whitespace
                content = re.sub(r'\s+', ' ', content).strip()
                
                # Create document
                doc = Document(
                    page_content=content,
                    metadata={
                        'source_type': 'web',
                        'url': url,
                        'title': title_text
                    }
                )
                self.documents.append(doc)
                print(f"    Article scraped: {title_text[:50]}...")
                
            except Exception as e:
                print(f"    Error scraping {url}: {e}")
        
        print(f"Web scraping completed")
    
    def process_video_transcripts(self, video_urls):
        """Process YouTube video transcripts (tokoh/expert opinions)"""
        print(f"\n🎥 Processing {len(video_urls)} video transcripts...")
        
        # Initialize whisper model for transcription
        try:
            whisper_model = whisper.load_model("base")
        except:
            print("Whisper not available, skipping video processing")
            return
        
        for i, url in enumerate(video_urls):
            try:
                print(f"  Processing video {i+1}: {url[:50]}...")
                
                # Download audio using youtube-dl
                ydl_opts = {
                    'format': 'bestaudio/best',
                    'outtmpl': f'/tmp/video_{i}.%(ext)s',
                    'quiet': True
                }
                
                with youtube_dl.YoutubeDL(ydl_opts) as ydl:
                    info = ydl.extract_info(url, download=True)
                    title = info.get('title', 'Unknown Video')
                    audio_file = f"/tmp/video_{i}.{info['ext']}"
                
                # Transcribe audio
                result = whisper_model.transcribe(audio_file)
                transcript = result['text']
                
                # Create document
                doc = Document(
                    page_content=transcript,
                    metadata={
                        'source_type': 'video',
                        'url': url,
                        'title': title
                    }
                )
                self.documents.append(doc)
                print(f"    Video transcribed: {title[:50]}...")
                
                # Clean up temp file
                os.remove(audio_file)
                
            except Exception as e:
                print(f"    Error processing video {url}: {e}")
        
        print(f"Video processing completed")
    
    def create_vector_store(self):
        """Create vector store from all documents using simple similarity search"""
        print(f"\nCreating vector store from {len(self.documents)} documents...")
        
        if not self.documents:
            print("No documents to process!")
            return False
        
        # Split documents into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100
        )
        
        all_chunks = []
        for doc in self.documents:
            chunks = text_splitter.split_documents([doc])
            all_chunks.extend(chunks)
        
        print(f"📝 Created {len(all_chunks)} text chunks")
        
        # Create embeddings for all chunks
        print("Creating embeddings...")
        self.chunk_texts = [chunk.page_content for chunk in all_chunks]
        self.chunk_metadata = [chunk.metadata for chunk in all_chunks]
        
        # Batch process embeddings
        batch_size = 10
        self.document_embeddings = []
        
        for i in range(0, len(self.chunk_texts), batch_size):
            batch = self.chunk_texts[i:i+batch_size]
            batch_embeddings = self.embeddings.embed_documents(batch)
            self.document_embeddings.extend(batch_embeddings)
            print(f"  Processed {min(i+batch_size, len(self.chunk_texts))}/{len(self.chunk_texts)} chunks")
        
        print("Vector store created successfully!")
        return True
    
    def retrieve_context(self, question, k=5):
        """Retrieve relevant context using simple cosine similarity"""
        if not self.document_embeddings:
            return ""
        
        try:
            print(f"Searching for relevant context...")
            
            # Get query embedding
            query_embedding = self.embeddings.embed_query(question)
            query_embedding = query_embedding.reshape(1, -1)
            
            # Calculate similarities
            doc_embeddings = np.array(self.document_embeddings)
            similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
            
            # Get top k most similar chunks
            top_indices = np.argsort(similarities)[::-1][:k]
            
            contexts = []
            for idx in top_indices:
                if similarities[idx] > 0.1:  # Minimum similarity threshold
                    metadata = self.chunk_metadata[idx]
                    source_type = metadata.get('source_type', 'unknown')
                    source_info = ""
                    
                    if source_type == 'pdf':
                        source_info = f"[PDF: {metadata.get('source_file', 'Unknown')}]"
                    elif source_type == 'web':
                        source_info = f"[Artikel: {metadata.get('title', 'Unknown')[:30]}...]"
                    elif source_type == 'video':
                        source_info = f"[Video: {metadata.get('title', 'Unknown')[:30]}...]"
                    
                    context_text = f"{source_info}\n{self.chunk_texts[idx]}"
                    contexts.append(context_text)
            
            print(f"Found {len(contexts)} relevant contexts")
            return "\n\n".join(contexts)
            
        except Exception as e:
            print(f"Error retrieving context: {e}")
            return ""

In [13]:
rag_system = MultiSourceRAG()

Loading embedding model: distilbert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [14]:
PDF_SOURCES = [
    "/kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf",
    "/kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf",
    "/kaggle/input/rag-pdf/naskah-akademik.pdf",
    "/kaggle/input/rag-pdf/wiraedsus2019-web.pdf"
]

WEB_SOURCES = [

]

VIDEO_SOURCES = [

]

In [15]:
# Process PDFs
for pdf_path in PDF_SOURCES:
    if os.path.exists(pdf_path):
        rag_system.process_pdf(pdf_path)
    else:
        print(f"PDF not found: {pdf_path}")

# Process web articles
if WEB_SOURCES:
    rag_system.scrape_web_articles(WEB_SOURCES)

# Process video transcripts
if VIDEO_SOURCES:
    rag_system.process_video_transcripts(VIDEO_SOURCES)



Processing PDF: /kaggle/input/rag-pdf/Permohonan_4288_8190_Permohonan_redact.pdf
PDF processed: 24 pages added

Processing PDF: /kaggle/input/rag-pdf/Petisi-Revisi-UU-TNI-.pdf
PDF processed: 11 pages added

Processing PDF: /kaggle/input/rag-pdf/naskah-akademik.pdf
PDF processed: 28 pages added

Processing PDF: /kaggle/input/rag-pdf/wiraedsus2019-web.pdf
PDF processed: 60 pages added


In [16]:
# Create vector store
if rag_system.documents:
    rag_system.create_vector_store()
else:
    print("No documents processed. Please check your data sources.")


Creating vector store from 123 documents...
📝 Created 766 text chunks
Creating embeddings...
  Processed 10/766 chunks
  Processed 20/766 chunks
  Processed 30/766 chunks
  Processed 40/766 chunks
  Processed 50/766 chunks
  Processed 60/766 chunks
  Processed 70/766 chunks
  Processed 80/766 chunks
  Processed 90/766 chunks
  Processed 100/766 chunks
  Processed 110/766 chunks
  Processed 120/766 chunks
  Processed 130/766 chunks
  Processed 140/766 chunks
  Processed 150/766 chunks
  Processed 160/766 chunks
  Processed 170/766 chunks
  Processed 180/766 chunks
  Processed 190/766 chunks
  Processed 200/766 chunks
  Processed 210/766 chunks
  Processed 220/766 chunks
  Processed 230/766 chunks
  Processed 240/766 chunks
  Processed 250/766 chunks
  Processed 260/766 chunks
  Processed 270/766 chunks
  Processed 280/766 chunks
  Processed 290/766 chunks
  Processed 300/766 chunks
  Processed 310/766 chunks
  Processed 320/766 chunks
  Processed 330/766 chunks
  Processed 340/766 chun

In [24]:
## Response Generation Function
def generate_response(question, use_rag=True):
    """Generate response using fine-tuned model + RAG"""
    
    # Retrieve context if RAG is enabled
    context = ""
    if use_rag and len(rag_system.document_embeddings) > 0:
        context = rag_system.retrieve_context(question)
        context_count = len([c for c in context.split('[') if c.strip()]) - 1
        print(f"\nRetrieved context from {max(0, context_count)} sources")
    
    # Create prompt
    if context:
        prompt = f"""Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut.

### Instruksi:
Anda adalah seorang ahli hukum TNI dengan pengetahuan mendalam tentang UU TNI. 
Jawablah pertanyaan berikut berdasarkan pengetahuan Anda yang telah di-fine-tune dengan pasal-pasal UU TNI, 
serta konteks tambahan dari berbagai sumber (buku, artikel, video ahli).

### Konteks dari berbagai sumber:
{context}

### Pertanyaan:
{question}

### Jawaban:
"""
    else:
        prompt = f"""### Instruksi:
Anda adalah seorang ahli hukum TNI dengan pengetahuan mendalam tentang UU TNI.
Jawablah pertanyaan berikut berdasarkan pengetahuan pasal-pasal UU TNI yang telah Anda pelajari.

### Pertanyaan:
{question}

### Jawaban:
"""
    
    # Generate response
    try:
        response = text_generator(
                    prompt,
                    max_new_tokens=1200,
                    min_new_tokens=50,                
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,                        
                    repetition_penalty=1.1,           
                    pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
                )
        
        # Extract answer
        generated_text = response[0]['generated_text']
        answer = generated_text.split("### Jawaban:")[-1].strip()
        
        # Clean up response
        answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
        
        return answer if answer else "Sorry, we can't give you a precise answer."
        
    except Exception as e:
        return f"Error generating response: {str(e)}"

In [25]:
## Testing Section
print("\n" + "="*60)
print("TESTING CHATBOT UU TNI")
print("="*60)

# Test questions
test_questions = [
    "Apa arti dari 'berada di bawah Presiden' dalam konteks TNI?",
    "Bagaimana hubungan TNI dengan Polri menurut UU TNI?",
    "Apa saja tugas pokok TNI berdasarkan undang-undang?",
    "Siapa yang berwenang mengangkat dan memberhentikan perwira tinggi TNI?",
    "Bagaimana mekanisme pengawasan terhadap TNI?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*20} TEST {i} {'='*20}")
    print(f"❓ Pertanyaan: {question}")
    print("\nJawaban (dengan RAG):")
    print("-" * 50)
    
    answer = generate_response(question, use_rag=True)
    print(answer)
    
    print("\n" + "="*60)

Both `max_new_tokens` (=1200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TESTING CHATBOT UU TNI

==================== TEST 1 ====================
❓ Pertanyaan: Apa arti dari 'berada di bawah Presiden' dalam konteks TNI?

Jawaban (dengan RAG):
--------------------------------------------------
Searching for relevant context...
Found 5 relevant contexts

Retrieved context from 4 sources


Both `max_new_tokens` (=1200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TNI adalah tentara Presiden, dimana 'berada di bawah Presiden' berarti mengikuti arahan langsung dari Presiden, meskipun dalam praktek umumnya ikonik yang ditampilkan adalah bentuk kekuasaan Presiden atas Tentara. Ini termasuk dalam asas-asas penting tentang hubungan antara Presiden dan TNI, yang tercantum dalam UU TNI dan doktrin TNI. Selain itu, 'berada di bawah Presiden' juga terkait dengan konsep kesetiaan dan loyalitas terhadap Negara Kesepuluhan Indonesia. (Selesai)
</think>

**Penyelesaian:**

Dalam konteks TNI, "berada di bawah Presiden" berarti Tentara Nasional Indonesia (TNI) merupakan tentara dari Presiden dan mengikuti arahan langsung dari Presiden. Meskipun dalam praktik umumnya, ikon kekuasaan Presiden atas Tentara ditampilkan melalui simbol tertentu, tetapi secara yuridis dan ideologi, TNI tetap merupakan tentara Presiden. Ini termasuk dalam asas penting hubungan antara Presiden dan TNI yang tercantum dalam UU TNI dan doktrin TNI. Selain itu, "berada di bawah Presiden" j

Both `max_new_tokens` (=1200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hubungan antara TNI dan Polri menurut UU TNI tercantum dalam Pasal 66 ayat f yang menyebutkan bahwa pertimbangan dapat atau tidak dapatnya suatu RUU masuk ke dalam prolegnas perubahan Undang-Undang tertentu (UU) masuk ke dalam prolegnas perubahan.

Selain itu, hubungan tersebut juga ditunjukkan dalam Pasal 66 ayat g yang merujuk pada peran dan fungsi kedua instansi sebagai satpam siapa saja yang berhak untuk melaporkan keadaan sehingga diperlukan pemeriksaan atau pengawasan terhadap aktivitasnya.

Dengan demikian hubungan TNI dan Polri saling mendukung dan melindungi kepentingan negara sesuai dengan azas-azas dan tujuan yang ditetapkan dalam UU TNI.

**Komentar:**

1. **Pertama:** Hubungan antara TNI dan Polri menurut UU TNI tercantum dalam Pasal 66 ayat f. Ini berarti informasi penting mengenai hubungan tersebut berada di situ.
   
2. **Kedua:** Selain itu, hubungan tersebut juga tercantum dalam Pasal 66 ayat g yang merujuk pada peran dan fungsi kedua instansi sebagai satpam siapa saj

Both `max_new_tokens` (=1200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TNI adalah tentara negara, yang memiliki tugas pokok untuk mempertahankan kewarganegaraan negara dan menyelamatkan negeri dari ancaman kedatangan, sesuai dengan tujuan dan prinsip yang diterima oleh UU TNI. TNI juga bertindak sebagai pelayan publik dengan memberikan layanan keamanan yang optimal kepada masyarakat. Dengan pendekatan holistik dan profesionalisme tinggi, TNI siap menghadapi segala ujian dan tantangan demi kepentingan Negara Republik Indonesia.

**Penjelasan jawaban:**
Dari dokumen-dokumen yang diberikan, terutama dokumen naskah akademik dan buku teks, diketahui bahwa TNI memiliki tugas utama untuk mempertahankan kewarganegaraan negara dan menyelamatkan negeri dari ancaman kedatangan. Selain itu, TNI juga bertindak sebagai pelayan publik dengan memberikan layanan keamanan yang optimal bagi masyarakat. Hal ini sesuai dengan tujuan dan prinsip yang diterima oleh UU TNI.

Selain itu, perlu dipahami bahwa undang-undang TNI tidak hanya mencakup aspek kebanggaan nasional melaink

Both `max_new_tokens` (=1200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pertama, saya akan mencari informasi mengenai siapakah yang berwenang mengangkat dan memberhentikan perwira tinggi TNI berdasarkan UU TNI.

Saya ingat bahwa UU TNI memiliki pasal yang menentukan prosedur pengangkatan dan pemberhentian perwira tinggi. Saya juga memahami bahwa ada komisi yang terlibat dalam proses ini.

Kemudian, saya akan mencari referensi tentang apakah Presiden memiliki kuasa untuk melakukan pengangkatan atau apakah ada mekanisme lainnya seperti Dewan Pertimbangan Perwira Tinggi yang terlibat.

Setelah itu, saya akan menyusun jawaban berdasarkan informasi yang ditemukan.
</think>

Berdasarkan Undang-Undang TNI (UU TNI) No. 14 Tahun 1951, termasuk revisinya yang telah ditetapkan melalui Keputusan Presiden Republik Indonesia No. 22 Tahun 2023, prosedur pengangkatan dan pemberhentian perwira tinggi TNI dibedakan menjadi beberapa tahapan dan dilakukan oleh mekanisme tertentu:

1. **Pengangkatan Perwira Tinggi TNI**:
   - Di luar batas masa jabatan yang ditentukan, anggota

In [ ]:
## Interactive Mode 
print("Type 'exit' to quit")

while True:
    try:
        user_input = input("\nYour Question: ")
        if user_input.lower() in ['exit', 'quit']:
            break

        if user_input.strip():
            print("\nAnswer:")
            print("-" * 40)
            response = generate_response(user_input, use_rag=True)
            print(response)

    except KeyboardInterrupt:
        break